# Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [6]:
import warnings
warnings.filterwarnings('ignore')

In [7]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [8]:
llm_model = "llama-3.3-70b-versatile"

In [ ]:
#!pip install pandas

In [9]:
import pandas as pd
df = pd.read_csv('Data.csv')

In [10]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [11]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [12]:
llm = ChatGroq(temperature=0.9, model_name=llm_model, api_key=os.environ["GROQ_API_KEY"])

In [13]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [14]:
chain = LLMChain(llm=llm, prompt=prompt)

In [15]:
product = "Queen Size Sheet Set"
chain.run(product)

"For a company that specializes in making Queen Size Sheet Sets, here are some name suggestions:\n\n1. **Royal Slumber**: This name plays on the idea of a queen-sized bed and evokes a sense of luxury and comfort.\n2. **QueenSheet Co.**: Simple and straightforward, this name clearly communicates the company's focus on queen-sized sheet sets.\n3. **Sleep Royale**: This name combines the ideas of sleep and royalty, suggesting a high-end product that will help customers sleep like royalty.\n4. **DreamWeaver Sheets**: This name has a poetic, dreamy quality to it, suggesting that the company's sheets will help customers weave their dreams into reality.\n5. **Queen Comfort**: This name emphasizes the comfort and coziness of the company's queen-sized sheet sets, which is likely to appeal to customers looking for a good night's sleep.\n6. **Regal Linens**: This name suggests a sense of luxury and high-end quality, which could help the company stand out in a crowded market.\n7. **Sheet Majesty**

## SimpleSequentialChain

In [16]:
from langchain_classic.chains import SimpleSequentialChain


In [17]:
llm = ChatGroq(temperature=0.9, model_name=llm_model, api_key=os.environ["GROQ_API_KEY"])
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [19]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [20]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [21]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Here are some name suggestions for a company that makes queen-size sheet sets:

1. **Regal Linens**: This name plays off the idea of queens and royalty, fitting for a company that specializes in queen-size sheet sets.
2. **DreamWeave**: This name suggests a high-quality, comfortable product that will help customers have a good night's sleep.
3. **SlumberSoft**: This name conveys the idea of soft, cozy sheets that will make customers feel relaxed and comfortable.
4. **QueenSize Comforts**: This name is straightforward and clearly communicates the company's focus on queen-size sheet sets.
5. **SleepHaven**: This name evokes a sense of a safe, cozy space where customers can rest and relax.
6. **LinenLuxury**: This name suggests a high-end product that is both comfortable and luxurious.
7. **CozyNights**: This name is simple and inviting, implying a warm and comfortable sleeping experience.
8. **Soft Sheets Co.**: This name is straightforward

"Soft, cozy queen-size sheet sets for a restful and comfortable night's sleep, every time, guaranteed always perfectly."

## SequentialChain

In [4]:
from langchain_classic.chains import SimpleSequentialChain


In [22]:
llm = ChatGroq(temperature=0.9, model_name=llm_model, api_key=os.environ["GROQ_API_KEY"])

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )


In [23]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )


In [24]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [27]:

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [26]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)

NameError: name 'SequentialChain' is not defined

In [28]:
review = df.Review[5]
overall_chain(review)

NameError: name 'overall_chain' is not defined

## Router Chain

In [29]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [30]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [31]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate

In [32]:
llm = ChatGroq(temperature=0, model_name=llm_model, api_key=os.environ["GROQ_API_KEY"])

In [33]:

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [34]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [35]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [36]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [37]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [38]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black body radiation is a fundamental concept in physics. It refers to the thermal electromagnetic radiation emitted by an object at a certain temperature. The term "black body" doesn\'t mean the object is actually black, but rather that it\'s an idealized object that absorbs all the electromagnetic radiation that falls on it, without reflecting or transmitting any of it.\n\nWhen an object is heated, its atoms or molecules vibrate and collide with each other, causing them to emit electromagnetic radiation across a wide range of wavelengths. The distribution of this radiation is dependent on the object\'s temperature, and it follows a specific pattern known as the black body spectrum.\n\nThe key characteristics of black body radiation are:\n\n1. It\'s dependent on the object\'s temperature, not its composition.\n2. The radiation is emitted across a wide range of wavelengths, from radio waves to gamma rays.\n3. The spectrum of the radiation follows a specific distribution, known as the 

In [39]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'To answer this question, let\'s break it down into its component parts.\n\nThe question is asking for the result of adding 2 and 2 together. \n\nThe component parts of this question are:\n1. The number 2\n2. The operation of addition (+)\n3. The number 2 (again, since we\'re adding 2 + 2)\n\nNow, let\'s answer the component parts:\n1. We know the value of the number 2.\n2. We know how to perform the operation of addition, which means combining two numbers to get their total or sum.\n3. We know the value of the second number 2.\n\nNow, let\'s put the component parts together to answer the broader question:\n2 + 2 = ?\n\nTo do this, we simply add the two numbers together:\n2 + 2 = 4\n\nTherefore, the answer to the question "what is 2 + 2" is 4.'

In [40]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'What is the role of DNA in cells?'}
> Finished chain.


'DNA (Deoxyribonucleic acid) plays a crucial role in cells, serving as the primary genetic material that contains the instructions for the development, growth, and function of an organism. The main roles of DNA in cells are:\n\n1. **Genetic Information Storage**: DNA stores the genetic information necessary for the development, growth, and function of an organism. It contains the instructions for the synthesis of proteins, which are the building blocks of all living things.\n2. **Transmission of Genetic Traits**: DNA is responsible for passing genetic traits from one generation to the next. The genetic information in DNA is transmitted from parents to offspring through the process of reproduction.\n3. **Protein Synthesis**: DNA provides the instructions for the synthesis of proteins, which are essential for various cellular functions, such as metabolism, growth, and repair.\n4. **Cellular Regulation**: DNA regulates cellular processes, such as cell division, growth, and differentiation

Reminder: Download your notebook to you local computer to save your work.